# HPGK AGENT v0.6 — Google Colab Trainer (FIXED)

Notebook này đã được sửa để **không chạy dây chuyền khi bước trước bị lỗi**.

Luồng: GitHub → Drive → kiểm tra GPU → dependencies → dataset → train/resume.

> **Quan trọng:** trước khi chạy ô kiểm tra GPU, vào **Runtime → Change runtime type → GPU**. Nếu dùng Colab Free, GPU có thể không được cấp.


In [ ]:
# 1) CẤU HÌNH
REPO_URL = "https://github.com/huynhphuocgiakhang3-arch/AIWEBSITE.git"
BRANCH = "main"
TRAIN_MINUTES = 45
EPOCHS = 1
VOCAB_SIZE = 4000

print("Repo:", REPO_URL)
print("Branch:", BRANCH)


In [ ]:
# 2) KẾT NỐI GOOGLE DRIVE
from google.colab import drive
drive.mount('/content/drive')

import os
PERSIST = '/content/drive/MyDrive/HPGK-AGENT'
CKPT_DIR = os.path.join(PERSIST, 'checkpoints')
os.makedirs(CKPT_DIR, exist_ok=True)
print('Persistent storage:', PERSIST)


In [ ]:
# 3) CLONE GITHUB AN TOÀN — public hoặc private
import os, subprocess, getpass, shutil, tempfile

TARGET = '/content/HPGK-AGENT'
shutil.rmtree(TARGET, ignore_errors=True)

# Thử clone public trước.
cmd = ['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, TARGET]
print('Đang clone branch:', BRANCH)
result = subprocess.run(cmd, text=True, capture_output=True)

if result.returncode != 0:
    print('Clone public thất bại.')
    print('Git:', (result.stderr or result.stdout).strip()[-1200:])
    print('Nếu repo là PRIVATE, nhập Personal Access Token có quyền đọc repo.')
    token = getpass.getpass('GitHub token (không hiển thị): ').strip()
    if not token:
        raise RuntimeError('Chưa nhập GitHub token. Không thể clone repository private.')

    # Dùng GIT_ASKPASS để token không nằm trong command-line/remote URL.
    askpass = '/tmp/hpgk-git-askpass.sh'
    with open(askpass, 'w') as f:
        f.write('#!/bin/sh
case "$1" in
  *Username*) printf "%s" "x-access-token" ;;
  *Password*) printf "%s" "$HPGK_GIT_TOKEN" ;;
esac
')
    os.chmod(askpass, 0o700)
    env = os.environ.copy()
    env['GIT_ASKPASS'] = askpass
    env['GIT_TERMINAL_PROMPT'] = '0'
    env['HPGK_GIT_TOKEN'] = token
    result = subprocess.run(cmd, text=True, capture_output=True, env=env)
    token = None
    env.pop('HPGK_GIT_TOKEN', None)
    try: os.remove(askpass)
    except FileNotFoundError: pass

    if result.returncode != 0:
        shutil.rmtree(TARGET, ignore_errors=True)
        raise RuntimeError('Clone GitHub thất bại. Kiểm tra token, quyền repo và tên branch.
' + (result.stderr or result.stdout)[-2000:])

# Luôn bỏ credential khỏi remote sau khi clone.
os.chdir(TARGET)
subprocess.run(['git', 'remote', 'set-url', 'origin', REPO_URL], check=False)
commit = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip()
print('✓ Clone thành công:', commit)
print('✓ Source:', TARGET)


In [ ]:
# 4) KIỂM TRA GPU TRƯỚC — không cho train nhầm bằng CPU
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError(
        'GPU CHƯA BẬT. Hãy chọn Runtime → Change runtime type → Hardware accelerator → GPU, ' +
        'sau đó chạy lại ô này từ đầu.'
    )
print('✓ GPU:', torch.cuda.get_device_name(0))


In [ ]:
# 5) CÀI DEPENDENCIES — repo phải tồn tại trước khi chạy ô này
from pathlib import Path
req = Path('ai/requirements.txt')
if not req.is_file():
    raise FileNotFoundError(
        f'Không tìm thấy {req}. Bước clone chưa thành công hoặc repo không đúng branch: {BRANCH}'
    )

# Colab đã có PyTorch; chỉ cài các dependency được khai báo nếu cần.
!pip -q install -r ai/requirements.txt
print('✓ requirements.txt:', req)


In [ ]:
# 6) KIỂM TRA DATASET
from pathlib import Path
files = list(Path('ai/data/raw').glob('**/*'))
docs = [p for p in files if p.is_file() and p.suffix.lower() in {'.txt','.md','.json','.jsonl'}]
print('Training documents:', len(docs))
for p in docs[:20]: print(' -', p)
if not docs:
    raise RuntimeError('Chưa có dữ liệu trong ai/data/raw. Hãy thêm .txt/.md/.json/.jsonl vào GitHub rồi chạy lại.')


In [ ]:
# 7) TRAIN / RESUME
import os, subprocess, sys
ckpt = os.path.join(CKPT_DIR, 'hpgk-v0.6.pt')
cmd = [sys.executable, 'ai/kaggle/train_kaggle.py',
       '--data_dir', 'ai/data/raw',
       '--epochs', str(EPOCHS),
       '--vocab_size', str(VOCAB_SIZE),
       '--max_minutes', str(TRAIN_MINUTES),
       '--checkpoint', ckpt]
if os.path.exists(ckpt):
    cmd.append('--resume')
    print('♻️ RESUME:', ckpt)
else:
    print('🆕 TRAIN FROM ZERO')
print('Starting training...')
subprocess.run(cmd, check=True)


In [ ]:
# 8) KIỂM TRA CHECKPOINT
import os, json
manifest = os.path.join(os.path.dirname(ckpt), 'manifest.json')
print('Checkpoint exists:', os.path.exists(ckpt), ckpt)
if not os.path.exists(ckpt):
    raise RuntimeError('Training kết thúc nhưng chưa thấy checkpoint. Kiểm tra log ô 7.')
if os.path.exists(manifest):
    print(json.dumps(json.load(open(manifest)), indent=2, ensure_ascii=False))
print('✓ Drive folder:', PERSIST)


## Lần sau

Mở lại notebook và chạy từ đầu. Nếu checkpoint còn trong `MyDrive/HPGK-AGENT/checkpoints`, ô 7 sẽ tự `RESUME`.

### Nếu gặp lỗi
- **Clone GitHub:** nếu repo private, ô 3 sẽ hỏi token; không dán token vào code.
- **GPU:** ô 4 sẽ dừng ngay nếu CUDA chưa bật, không chạy training bằng CPU.
- **requirements:** ô 5 chỉ chạy sau khi clone thành công.
- **dataset:** ô 6 chỉ chạy khi `ai/data/raw` có dữ liệu.
